In [1]:
# load “Imports + config” notebook
%run ./0_setup.ipynb

In [2]:
def average_soil_layers(tif_files, out_tif, nodata_value=-9999):
    """
    Average multiple depth-specific GeoTIFFs into one GeoTIFF.
    Assumes all rasters are already aligned.
    """

    # Precondition check
    with rasterio.open(tif_files[0]) as ref:
        ref_crs = ref.crs
        ref_res = ref.res
        ref_bounds = ref.bounds
        ref_transform = ref.transform

    for tif in tif_files[1:]:
        with rasterio.open(tif) as src:
            if not (
                src.crs == ref_crs and
                src.res == ref_res and
                src.bounds == ref_bounds and
                src.transform == ref_transform
            ):
                raise ValueError(f"Raster mismatch detected: {tif}")

    # Read rasters
    with rasterio.open(tif_files[0]) as src0:
        meta = src0.meta.copy()
        data_stack = []

        for tif in tif_files:
            with rasterio.open(tif) as src:
                band = src.read(1).astype("float32")

                if src.nodata is not None:
                    band = np.where(band == src.nodata, np.nan, band)

                data_stack.append(band)

    # Average
    stack = np.stack(data_stack, axis=0)
    mean_raster = np.nanmean(stack, axis=0)
    weights = np.array([5, 10, 15])

    # depth-weighted average
    weighted_mean = (
        np.nansum(stack * weights[:, None, None], axis=0) / np.sum(weights)
    )
    
    mean_raster = np.where(np.isnan(weighted_mean), nodata_value, weighted_mean)

    # Write output
    meta.update(dtype="float32", count=1, nodata=nodata_value)

    os.makedirs(os.path.dirname(out_tif), exist_ok=True)

    with rasterio.open(out_tif, "w", **meta) as dst:
        dst.write(mean_raster, 1)

    print("Saved:", out_tif)

In [3]:
soil_layers = {
    "BulkDensity": [
        "BulkDensity0-5cm.tif",
        "BulkDensity5-15cm.tif",
        "BulkDensity15-30cm.tif"
    ],
    "ClayContent": [
        "ClayContent0-5cm.tif",
        "ClayContent5-15cm.tif",
        "ClayContent15-30cm.tif"
    ],
    "Sand": [
        "Sand0-5cm.tif",
        "Sand5-15cm.tif",
        "Sand15-30cm.tif"
    ],
    "SOC": [
        "SOC0-5cm.tif",
        "SOC5-15cm.tif",
        "SOC15-30cm.tif"
    ],
    "pH": [
        "pH0-5cm.tif",
        "pH5-15cm.tif",
        "pH15-30cm.tif"
    ],
    "CEC": [
        "CEC0-5cm.tif",
        "CEC5-15cm.tif",
        "CEC15-30cm.tif"
    ],
    "Nitrogen": [
        "Nitrogen0-5cm.tif",
        "Nitrogen5-15cm.tif",
        "Nitrogen15-30cm.tif"
    ],
    "CoarseFragments": [
        "CoarseFragments0-5cm.tif",
        "CoarseFragments5-15cm.tif",
        "CoarseFragments15-30cm.tif"
    ]
}


In [4]:
for soil, layers in soil_layers.items():
    tif_files = [os.path.join(SOIL_PATH , f) for f in layers]
    out_tif = os.path.join(SOIL_PATH , f"{soil}0-30cm.tif")

    average_soil_layers(tif_files, out_tif)

Saved: /Users/kc/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/BulkDensity0-30cm.tif
Saved: /Users/kc/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/ClayContent0-30cm.tif
Saved: /Users/kc/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/Sand0-30cm.tif
Saved: /Users/kc/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/SOC0-30cm.tif
Saved: /Users/kc/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/pH0-30cm.tif
Saved: /Users/kc/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/CEC0-30cm.tif
Saved: /Users/kc/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/Nitrogen0-30cm.tif
Saved: /Users/kc/Desktop/CERDIProject/TermiteMound/Script/RawData/ShapeFile/SoilGrid_Cambodia/CoarseFragments0-30cm.tif
